# Homework: Vector Search

In [1]:
# import packages
import requests

import numpy as np
import pandas as pd

from fastembed import TextEmbedding
from qdrant_client import QdrantClient, models
from qdrant_client.models import Distance, VectorParams, PointStruct

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Q1. Embedding the query

Embed the query: `'I just discovered the course. Can I join now?'`.
Use the `'jinaai/jina-embeddings-v2-small-en'` model. 

You should get a numpy array of size 512.

What's the minimal value in this array?

* -0.51
* -0.11
* 0
* 0.51


### Q1 Answer

-0.11

In [2]:
# working
embedder = TextEmbedding('jinaai/jina-embeddings-v2-small-en')

qns = "I just discovered the course. Can I join now?"
qns_embed, = list(embedder.embed(qns))

In [3]:
qns_embed.min()

np.float64(-0.11726375058706588)

## Q2. Cosine similarity with another vector

Now let's embed this document:

```python
doc = 'Can I still join the course after the start date?'
```

What's the cosine similarity between the vector for the query
and the vector for the document?

* 0.3
* 0.5
* 0.7
* 0.9


### Q2 Answer
0.9

In [4]:
# working
np.linalg.norm(qns_embed)
qns_embed.dot(qns_embed)

np.float64(1.0)

np.float64(1.0)

In [5]:
doc = "Can I still join the course after the start date?"
doc_embed, = list(embedder.embed(doc))

In [6]:
qns_embed.dot(doc_embed)

np.float64(0.9008528887793987)

## Q3. Ranking by cosine

For Q3 and Q4 we will use these documents:

```python
documents = [{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
  'section': 'General course-related questions',
  'question': 'Course - What can I do before the course starts?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.',
  'section': 'General course-related questions',
  'question': 'How can we contribute to the course?',
  'course': 'data-engineering-zoomcamp'}]
```

Compute the embeddings for the text field, and compute the 
cosine between the query vector and all the documents.

What's the document index with the highest similarity? (Indexing starts from 0):

- 0
- 1
- 2
- 3
- 4

Hint: if you put all the embeddings of the text field in one matrix `V` (a single 2-dimensional numpy array), then
computing the cosine becomes a matrix multiplication:

```python
V.dot(q)
```

### Q3 Answer

Index 1

In [7]:
# workings
documents = [{'text': "Yes, even if you don't register, you're still eligible to submit the homeworks.\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything for the last minute.",
  'section': 'General course-related questions',
  'question': 'Course - Can I still join the course after the start date?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Yes, we will keep all the materials after the course finishes, so you can follow the course at your own pace after it finishes.\nYou can also continue looking at the homeworks and continue preparing for the next cohort. I guess you can also start working on your final capstone project.',
  'section': 'General course-related questions',
  'question': 'Course - Can I follow the course after it finishes?',
  'course': 'data-engineering-zoomcamp'},
 {'text': "The purpose of this document is to capture frequently asked technical questions\nThe exact day and hour of the course will be 15th Jan 2024 at 17h00. The course will start with the first  “Office Hours'' live.1\nSubscribe to course public Google Calendar (it works from Desktop only).\nRegister before the course starts using this link.\nJoin the course Telegram channel with announcements.\nDon’t forget to register in DataTalks.Club's Slack and join the channel.",
  'section': 'General course-related questions',
  'question': 'Course - When will the course start?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'You can start by installing and setting up all the dependencies and requirements:\nGoogle cloud account\nGoogle Cloud SDK\nPython 3 (installed with Anaconda)\nTerraform\nGit\nLook over the prerequisites and syllabus to see if you are comfortable with these subjects.',
  'section': 'General course-related questions',
  'question': 'Course - What can I do before the course starts?',
  'course': 'data-engineering-zoomcamp'},
 {'text': 'Star the repo! Share it with friends if you find it useful ❣️\nCreate a PR if you see you can improve the text or the structure of the repository.',
  'section': 'General course-related questions',
  'question': 'How can we contribute to the course?',
  'course': 'data-engineering-zoomcamp'}]

In [8]:
documents_df = pd.DataFrame(documents)
documents_df

,text,section,question,course
0,"Yes, even if you don't register, you're still ...",General course-related questions,Course - Can I still join the course after the...,data-engineering-zoomcamp
1,"Yes, we will keep all the materials after the ...",General course-related questions,Course - Can I follow the course after it fini...,data-engineering-zoomcamp
2,The purpose of this document is to capture fre...,General course-related questions,Course - When will the course start?,data-engineering-zoomcamp
3,You can start by installing and setting up all...,General course-related questions,Course - What can I do before the course starts?,data-engineering-zoomcamp
4,Star the repo! Share it with friends if you fi...,General course-related questions,How can we contribute to the course?,data-engineering-zoomcamp


In [9]:
# embed text column
vec_text = list(embedder.embed(documents_df.text))
vec_text = np.array(vec_text)
vec_text

array([[-0.02495246, -0.03964539, -0.00437672, ...,  0.06779867,
         0.0377203 , -0.00470726],
       [-0.05947089, -0.08523985,  0.0129209 , ...,  0.09599707,
         0.0542084 , -0.0002946 ],
       [-0.06846453, -0.04079098,  0.04998119, ...,  0.0657807 ,
         0.02872073, -0.01115214],
       [-0.04640507, -0.02555228,  0.02241224, ...,  0.03410311,
         0.06813594, -0.00640331],
       [-0.05394913, -0.04693815,  0.00794724, ...,  0.05402997,
         0.03033385, -0.01254517]], shape=(5, 512))

In [10]:
similarity = vec_text.dot(qns_embed)
similarity
similarity.argmax()

array([0.76296848, 0.81823785, 0.80853969, 0.71330787, 0.73044991])

np.int64(1)

## Q4. Ranking by cosine, version two

Now let's calculate a new field, which is a concatenation of
`question` and `text`:

```python
full_text = doc['question'] + ' ' + doc['text']
``` 

Embed this field and compute the cosine between it and the
query vector. What's the highest scoring document?

- 0
- 1
- 2
- 3
- 4

### Q4 Answer
Index 0

Text-only embedding is like searching by body content only — you find what's written similarly, not necessarily what's topically correct. When the question field is concatenated in. Index 0's question literally contains "join the course" — the exact concept from the query "Can I join now?". The question field acts as a semantic title, anchoring the embedding to the document's intended topic. The model now recognizes: this document is about joining a course, and scores it highest.

| Field(s) embedded | What it captures           | Winner                    |
|-------------------|----------------------------|---------------------------|
| text only         | What the answer says       | Index 1 (textual overlap) |
| question + text   | What the document is about | Index 0 (topical match)   |

Adding the question gives the embedding a stronger signal about the document's intended meaning, not just its surface text.

In [11]:
# workings
documents_df['full_text'] = documents_df['question'] + ' ' + documents_df['text']
documents_df

,text,section,question,course,full_text
0,"Yes, even if you don't register, you're still ...",General course-related questions,Course - Can I still join the course after the...,data-engineering-zoomcamp,Course - Can I still join the course after the...
1,"Yes, we will keep all the materials after the ...",General course-related questions,Course - Can I follow the course after it fini...,data-engineering-zoomcamp,Course - Can I follow the course after it fini...
2,The purpose of this document is to capture fre...,General course-related questions,Course - When will the course start?,data-engineering-zoomcamp,Course - When will the course start? The purpo...
3,You can start by installing and setting up all...,General course-related questions,Course - What can I do before the course starts?,data-engineering-zoomcamp,Course - What can I do before the course start...
4,Star the repo! Share it with friends if you fi...,General course-related questions,How can we contribute to the course?,data-engineering-zoomcamp,How can we contribute to the course? Star the ...


In [12]:
# embed question + document text together
vec_text_2 = list(embedder.embed(documents_df.full_text))
vec_text_2 = np.array(vec_text_2)

In [13]:
similarity = vec_text_2.dot(qns_embed)
similarity
similarity.argmax()

array([0.85145433, 0.84365943, 0.8408287 , 0.77551578, 0.80860078])

np.int64(0)

## Q5. Selecting the embedding model

Now let's select a smaller embedding model.
What's the smallest dimensionality for models in fastembed?

- 128
- 256
- 384
- 512

One of these models is `BAAI/bge-small-en`. Let's use it.


### Q5 Answer

384 dimensions

In [14]:
# working
models = TextEmbedding.list_supported_models()
models_df = pd.DataFrame(models)
models_df

,model,sources,model_file,description,license,size_in_GB,additional_files,dim,tasks
0,BAAI/bge-base-en,"{'hf': 'Qdrant/fast-bge-base-en', 'url': 'http...",model_optimized.onnx,"Text embeddings, Unimodal (text), English, 512...",mit,0.420,[],768,{}
1,BAAI/bge-base-en-v1.5,"{'hf': 'qdrant/bge-base-en-v1.5-onnx-q', 'url'...",model_optimized.onnx,"Text embeddings, Unimodal (text), English, 512...",mit,0.210,[],768,{}
2,BAAI/bge-large-en-v1.5,"{'hf': 'qdrant/bge-large-en-v1.5-onnx', 'url':...",model.onnx,"Text embeddings, Unimodal (text), English, 512...",mit,1.200,[],1024,{}
3,BAAI/bge-small-en,"{'hf': 'Qdrant/bge-small-en', 'url': 'https://...",model_optimized.onnx,"Text embeddings, Unimodal (text), English, 512...",mit,0.130,[],384,{}
4,BAAI/bge-small-en-v1.5,"{'hf': 'qdrant/bge-small-en-v1.5-onnx-q', 'url...",model_optimized.onnx,"Text embeddings, Unimodal (text), English, 512...",mit,0.067,[],384,{}
5,BAAI/bge-small-zh-v1.5,"{'hf': 'Qdrant/bge-small-zh-v1.5', 'url': 'htt...",model_optimized.onnx,"Text embeddings, Unimodal (text), Chinese, 512...",mit,0.090,[],512,{}
6,mixedbread-ai/mxbai-embed-large-v1,"{'hf': 'mixedbread-ai/mxbai-embed-large-v1', '...",onnx/model.onnx,"Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.640,[],1024,{}
7,snowflake/snowflake-arctic-embed-xs,"{'hf': 'snowflake/snowflake-arctic-embed-xs', ...",onnx/model.onnx,"Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.090,[],384,{}
8,snowflake/snowflake-arctic-embed-s,"{'hf': 'snowflake/snowflake-arctic-embed-s', '...",onnx/model.onnx,"Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.130,[],384,{}
9,snowflake/snowflake-arctic-embed-m,"{'hf': 'Snowflake/snowflake-arctic-embed-m', '...",onnx/model.onnx,"Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.430,[],768,{}


In [15]:
# filtering for only models that has the smallest dimensions
models_df[models_df.dim == models_df.dim.min()]

,model,sources,model_file,description,license,size_in_GB,additional_files,dim,tasks
3,BAAI/bge-small-en,"{'hf': 'Qdrant/bge-small-en', 'url': 'https://...",model_optimized.onnx,"Text embeddings, Unimodal (text), English, 512...",mit,0.130,[],384,{}
4,BAAI/bge-small-en-v1.5,"{'hf': 'qdrant/bge-small-en-v1.5-onnx-q', 'url...",model_optimized.onnx,"Text embeddings, Unimodal (text), English, 512...",mit,0.067,[],384,{}
7,snowflake/snowflake-arctic-embed-xs,"{'hf': 'snowflake/snowflake-arctic-embed-xs', ...",onnx/model.onnx,"Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.090,[],384,{}
8,snowflake/snowflake-arctic-embed-s,"{'hf': 'snowflake/snowflake-arctic-embed-s', '...",onnx/model.onnx,"Text embeddings, Unimodal (text), English, 512...",apache-2.0,0.130,[],384,{}
14,sentence-transformers/all-MiniLM-L6-v2,"{'hf': 'qdrant/all-MiniLM-L6-v2-onnx', 'url': ...",model.onnx,"Text embeddings, Unimodal (text), English, 256...",apache-2.0,0.090,[],384,{}
26,sentence-transformers/paraphrase-multilingual-...,{'hf': 'qdrant/paraphrase-multilingual-MiniLM-...,model_optimized.onnx,"Text embeddings, Unimodal (text), Multilingual...",apache-2.0,0.220,[],384,{}


## Q6. Indexing with qdrant (2 points)

For the last question, we will use more documents.

We will select only FAQ records from our ml zoomcamp:

```python
import requests 

docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()


documents = []

for course in documents_raw:
    course_name = course['course']
    if course_name != 'machine-learning-zoomcamp':
        continue

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)
```

Add them to qdrant using the model form Q5.

When adding the data, use both question and answer fields:

```python
text = doc['question'] + ' ' + doc['text']
```

After the data is inserted, use the question from Q1 for querying the collection.

What's the highest score in the results?
(The score for the first returned record):

- 0.97
- 0.87
- 0.77
- 0.67


### Q6 Answer

Highest score: 0.87 (score=0.87031)

In [16]:
# workings
docs_url = 'https://github.com/alexeygrigorev/llm-rag-workshop/raw/main/notebooks/documents.json'
docs_response = requests.get(docs_url)
documents_raw = docs_response.json()

documents = []

for course in documents_raw:
    course_name = course['course']
    if course_name != 'machine-learning-zoomcamp':
        continue

    for doc in course['documents']:
        doc['course'] = course_name
        documents.append(doc)

documents_df = pd.DataFrame(documents)
documents_df.head()

,text,section,question,course
0,Machine Learning Zoomcamp FAQ\nThe purpose of ...,General course-related questions,How do I sign up?,machine-learning-zoomcamp
1,"The course videos are pre-recorded, you can st...",General course-related questions,Is it going to be live? When?,machine-learning-zoomcamp
2,"Everything is recorded, so you won’t miss anyt...",General course-related questions,What if I miss a session?,machine-learning-zoomcamp
3,The bare minimum. The focus is more on practic...,General course-related questions,How much theory will you cover?,machine-learning-zoomcamp
4,Yes! We'll cover some linear algebra in the co...,General course-related questions,I don't know math. Can I take the course?,machine-learning-zoomcamp


In [17]:
# use the model in Q5
embedder_bge_small = TextEmbedding('BAAI/bge-small-en')

In [18]:
# add new concat column in dataframe
documents_df['full_text'] = documents_df.question + ' ' + documents_df.text
documents_df.shape
documents_df.head()

(375, 5)

,text,section,question,course,full_text
0,Machine Learning Zoomcamp FAQ\nThe purpose of ...,General course-related questions,How do I sign up?,machine-learning-zoomcamp,How do I sign up? Machine Learning Zoomcamp FA...
1,"The course videos are pre-recorded, you can st...",General course-related questions,Is it going to be live? When?,machine-learning-zoomcamp,Is it going to be live? When? The course video...
2,"Everything is recorded, so you won’t miss anyt...",General course-related questions,What if I miss a session?,machine-learning-zoomcamp,What if I miss a session? Everything is record...
3,The bare minimum. The focus is more on practic...,General course-related questions,How much theory will you cover?,machine-learning-zoomcamp,How much theory will you cover? The bare minim...
4,Yes! We'll cover some linear algebra in the co...,General course-related questions,I don't know math. Can I take the course?,machine-learning-zoomcamp,I don't know math. Can I take the course? Yes!...


In [19]:
embeddings = list(embedder_bge_small.embed(documents_df.full_text))
embeddings = np.array(embeddings)
embeddings.shape 

(375, 384)

In [20]:
qd_client = QdrantClient(":memory:") # use in memory instead of starting docker
collection_name = "llmzoomcamp-homework"

# create a new collection
qd_client.create_collection(
    collection_name = collection_name,
    vectors_config = VectorParams(
        size=384, 
        distance=Distance.COSINE
    ),
)


points = [
    PointStruct(id=idx, 
                vector=embeddings[idx].tolist(),
                payload=doc)
    for idx, doc in enumerate(documents)
]

qd_client.upsert(collection_name=collection_name, points=points)

True

UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [21]:
q1 = "I just discovered the course. Can I join now?"

q1_embed_q5, = list(embedder_bge_small.embed(q1))

results = qd_client.query_points(
    collection_name = collection_name,
    query = q1_embed_q5.tolist(),
    limit = 5,
)

In [22]:
print(results)

points=[ScoredPoint(id=14, version=0, score=0.870317263743601, payload={'text': 'Yes, you can. You won’t be able to submit some of the homeworks, but you can still take part in the course.\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers’ Projects by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.', 'section': 'General course-related questions', 'question': 'The course has already started. Can I still join it?', 'course': 'machine-learning-zoomcamp'}, vector=None, shard_key=None, order_value=None), ScoredPoint(id=6, version=0, score=0.8691884621393949, payload={'text': 'Approximately 4 months, but may take more if you want to do some extra activities (an extra project, an article, etc)', 'section': 'General course-related questions', 'question': 'How long is the course?', 'course': 'machine-learning-zoomcamp'}, vector=None, shard_ke